# Environment & inference

## Mục tiêu
Tensor là mảng số; device là nơi tính toán; weights là tham số đã học. PIL dùng RGB, ndarray của Ultralytics dùng BGR. Ảnh fixture trống chỉ kiểm đường chạy, không đo độ chính xác.

## Setup
Chạy từ repo đã clone ở revision bạn ghi nhận. Local dùng venv API; Colab xem docs/RESEARCH.md. Không tự cài dependency hoặc tải dữ liệu khi Run all.

In [1]:
from pathlib import Path
import os, sys, json, tempfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ai/research").is_dir()), None)
assert ROOT is not None, "Clone repo và chạy từ repo/notebooks"
sys.path.insert(0, str(ROOT))
os.environ["YOLO_AUTOINSTALL"] = "false"
from ai.research.cli import environment
print(json.dumps(environment(), indent=2))

{
  "python": "3.11.9",
  "platform": "Windows-10-10.0.26200-SP0",
  "versions": {
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "ultralytics": "8.4.162",
    "numpy": "1.26.4",
    "pillow": "10.3.0"
  },
  "cuda_available": true,
  "gpu": "NVIDIA GeForce RTX 3060"
}


## Các bước
Đọc cấu hình trước khi thực thi; các thao tác tốn tài nguyên mặc định tắt.

In [2]:
from PIL import Image
import time, statistics
model_path = ROOT / "apps/api/models/yolov8x-pose.pt"
if not model_path.exists():
    print("SKIP inference: cung cấp weights local đã duyệt license. Không tự download.")
else:
    import torch
    from ultralytics import YOLO
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = YOLO(str(model_path))
    fixture = Image.new("RGB", (640, 640), "white")
    durations = []
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    for index in range(5):
        start = time.perf_counter()
        result = model.predict(fixture, imgsz=640, device=device, quantize=16 if device != "cpu" else None, verbose=False)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        durations.append((time.perf_counter() - start) * 1000)
    print({"input": "synthetic blank RGB 640x640", "persons": len(result[0].boxes),
           "mean_ms_after_2_warmups": statistics.mean(durations[2:]),
           "actual_dtype": str(next(model.predictor.model.model.parameters()).dtype),
           "peak_allocated_MiB": torch.cuda.max_memory_allocated() / 1024**2 if device != "cpu" else None,
           "accuracy_evaluated": False})
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

{'input': 'synthetic blank RGB 640x640', 'persons': 0, 'mean_ms_after_2_warmups': 33.69133333277811, 'actual_dtype': 'torch.float16', 'peak_allocated_MiB': 268.279296875, 'accuracy_evaluated': False}


## Kiểm tra
Output ghi rõ thao tác thực sự chạy và thao tác bị bỏ qua. Thiếu dữ liệu không được thay bằng số giả. Khi thay dữ liệu/cấu hình, restart kernel và Run all.

## Bước tiếp theo
Đổi sang một ảnh được phép dùng và xem keypoints/mask. Không xem zero detections của ảnh trống là lỗi model.